In [1]:
# Import required libraries
import os
import pandas as pd
from utils import log_standard_scale, peek

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_SDY2941.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,SDY2941.SUB395006,7,ENSG00000000003,14.0,0.197000,Whole blood
1,SDY2941.SUB395006,3,ENSG00000000003,3.0,0.049508,Whole blood
2,SDY2941.SUB395006,0,ENSG00000000003,1.0,0.013695,Whole blood
3,SDY2941.SUB395005,90,ENSG00000000003,5.0,0.073354,Whole blood
4,SDY2941.SUB395005,90,ENSG00000000003,2.0,0.030201,Whole blood


In [3]:
# raw_count has actual values (no nulls); material is uniformly 'Whole blood'
# SDY2941 has 6 timepoints: [0, 3, 7, 28, 85, 90]
# Drop material before pivoting; drop raw_count to keep tpm_count only, consistent with 2019/2020 pipeline
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Whole blood']
Length: 1, dtype: str
timepoints: [np.int64(0), np.int64(3), np.int64(7), np.int64(28), np.int64(85), np.int64(90)]
participants: 53


In [4]:
# Drop material (entirely 'Whole blood'); drop raw_count to match pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2941.SUB395006,7,ENSG00000000003,0.197000
1,SDY2941.SUB395006,3,ENSG00000000003,0.049508
2,SDY2941.SUB395006,0,ENSG00000000003,0.013695
3,SDY2941.SUB395005,90,ENSG00000000003,0.073354
4,SDY2941.SUB395005,90,ENSG00000000003,0.030201


In [5]:
df['timepoint'].unique()

array([ 7,  3,  0, 90, 85, 28])

In [6]:
# Filter to only timepoints 0, 7. Timepoints 3, 85, 90 not in the challenge set
df = df[df['timepoint'].isin([0, 7])]

# Pivot: each row is a participant_id, each gene+timepoint becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)
df_pivot.head(1)

(53, 164707)


,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d28_ENSG00000310527,d28_ENSG00000310529,d28_ENSG00000310530,d28_ENSG00000310532,d28_ENSG00000310533,d28_ENSG00000310534,d28_ENSG00000310535,d28_ENSG00000310536,d28_ENSG00000310537,d28_ENSG00000310539
0,SDY2941.SUB394970,0.076436,0.0,28.472615,7.858272,1.775265,511.323918,1.13743,8.304955,5.083847,...,58.187897,0.0,0.0,0.0,5.660007,1.207308,4.076846,0.322652,3.427343,0.966716


In [7]:
peek(df_pivot)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d28_ENSG00000310527,d28_ENSG00000310529,d28_ENSG00000310530,d28_ENSG00000310532,d28_ENSG00000310533,d28_ENSG00000310534,d28_ENSG00000310535,d28_ENSG00000310536,d28_ENSG00000310537,d28_ENSG00000310539
0,SDY2941.SUB394970,0.076436,0.0,28.472615,7.858272,1.775265,511.323918,1.13743,8.304955,5.083847,...,58.187897,0.0,0.0,0.0,5.660007,1.207308,4.076846,0.322652,3.427343,0.966716


In [ ]:
df_pivot = log_standard_scale(df_pivot)
peek(df_pivot)

In [8]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_SDY2941_cleaned.csv', index=False)